In [1]:
# =====================================
# Customer Segmentation Project
# Author: Rizwan Hussain
# =====================================

import warnings
warnings.filterwarnings("ignore") 

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.2f}".format)

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [2]:
# Load Dataset

df = pd.read_csv(r"C:\Users\Rizwan Hussain\Downloads\Compressed\online_retail_II.csv")

print("Dataset Loaded Successfully")
df.head()

Dataset Loaded Successfully


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.00,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.00,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.00,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.00,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.00,United Kingdom


In [3]:
# Dataset Information and shape details.

print("Rows :", df.shape[0])
print("Columns :", df.shape[1],'\n') # shape of the dataset

print("Column Names\n",df.columns,'\n') # column names

print("Information\n",df.info(),'\n') # information about the dataset

print("Statistical Summary:\n",df.describe(include='all'),'\n') # statistical summary of the dataset

print("Missing Values:\n", df.isnull().sum().sum(),'\n') # check for missing values

print("Duplicated Rows:\n", df.duplicated().sum(),'\n') # check for duplicated rows

print("Unique Values:\n", df.nunique(),'\n') # check for unique values in each column

print("Check DataTypes:\n", df.dtypes,'\n') # check for data types of each column

Rows : 1067371
Columns : 8 

Column Names
 Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country'],
      dtype='str') 

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  str    
 1   StockCode    1067371 non-null  str    
 2   Description  1062989 non-null  str    
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  str    
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 136.6 MB
Information
 None 

Statistical Summary:
         Invoice StockCode                         Description   Quantity  \
count   1067371   1067371                             1062989 1067371.00   
unique    53628      5305

In [4]:
# Business Rule Validation

print("Customer IDs missing:", df['Customer ID'].isnull().sum()) 

print("Negative Quantities:", (df['Quantity'] < 0).sum()) 

print("Zero Prices", (df['Price'] <= 0).sum())

print("Missing Description", df['Description'].isnull().sum())

Customer IDs missing: 243007
Negative Quantities: 22950
Zero Prices 6207
Missing Description 4382


In [5]:
# Executive Summary

print("="*60)
print("DATA PROFILING SUMMARY")
print("="*60)

print(f"Rows                 : {df.shape[0]}")
print(f"Columns              : {df.shape[1]}")
print(f"Missing Values       : {df.isnull().sum().sum()}")
print(f"Duplicate Records    : {df.duplicated().sum()}")
print(f"Unique Customers     : {df['Customer ID'].nunique()}")
print(f"Unique Products      : {df['StockCode'].nunique()}")
print(f"Unique Countries     : {df['Country'].nunique()}")

print("="*60)

DATA PROFILING SUMMARY
Rows                 : 1067371
Columns              : 8
Missing Values       : 247389
Duplicate Records    : 34335
Unique Customers     : 5942
Unique Products      : 5305
Unique Countries     : 43


In [6]:
# =====================================
# Create Working Copy of Dataset
# =====================================

clean_df = df.copy()

print("Working copy created successfully.")

Working copy created successfully.


In [7]:
# Record Initial Dataset Size
initial_rows = clean_df.shape[0]

print(f"Initial Rows: {initial_rows:,}")

Initial Rows: 1,067,371


In [8]:
# Remove Duplicates
duplicate_count = clean_df.duplicated().sum()

print(f"Duplicate Records Found: {duplicate_count:,}")

clean_df.drop_duplicates(inplace=True)
print("Duplicate Records Removed Successfully.")

print(f"Remaining Rows: {clean_df.shape[0]:,}")

Duplicate Records Found: 34,335
Duplicate Records Removed Successfully.
Remaining Rows: 1,033,036


In [9]:
# Missing Customer IDs
missing_customer = clean_df["Customer ID"].isnull().sum()

print(f"Missing Customer IDs: {missing_customer:,}")

Missing Customer IDs: 235,151


In [10]:
clean_df = clean_df.dropna(subset=["Customer ID"])

clean_df["Customer ID"].isnull().sum()

np.int64(0)

In [11]:
# Missing Descriptions
clean_df["Description"].isnull().sum()

np.int64(0)

In [12]:
# Cancelled Orders
cancelled_orders = clean_df[
    clean_df["Invoice"].astype(str).str.startswith("C")
]

print(f"Cancelled Orders: {len(cancelled_orders):,}")

# Remove Cancelled Orders
clean_df = clean_df[
    ~clean_df["Invoice"].astype(str).str.startswith("C")
]
print(f"Cancelled Orders: {len(cancelled_orders):,}")

Cancelled Orders: 18,390
Cancelled Orders: 18,390


In [13]:
# Negative Quantities
(clean_df["Quantity"] <= 0).sum()

np.int64(0)

In [14]:
# Invalid Prices
(clean_df["Price"] <= 0).sum()

np.int64(70)

In [15]:
# Remove Invalid Prices
clean_df = clean_df[
    clean_df["Price"] > 0
]

print(f"Invalid Prices: {clean_df['Price'].le(0).sum():,}")

Invalid Prices: 0


In [16]:
# Convert Customer ID to Integer
clean_df["Customer_ID"] = clean_df["Customer ID"].astype(int)

In [17]:
# Create Revenue Column
clean_df["Revenue"] = (
    clean_df["Quantity"] *
    clean_df["Price"]
)

print("Revenue Columns:", clean_df["Revenue"].duplicated().sum())

Revenue Columns: 775506


In [18]:
# Final Validation
print("=" * 60)
print("DATA CLEANING SUMMARY")
print("=" * 60)

print(f"Rows Remaining          : {clean_df.shape[0]:,}")
print(f"Missing Values          : {clean_df.isnull().sum().sum():,}")
print(f"Duplicate Records       : {clean_df.duplicated().sum():,}")
print(f"Negative Quantities     : {(clean_df['Quantity'] <= 0).sum():,}")
print(f"Invalid Prices          : {(clean_df['Price'] <= 0).sum():,}")
print(f"Missing Customer IDs    : {clean_df['Customer ID'].isnull().sum():,}")

print("=" * 60)

DATA CLEANING SUMMARY
Rows Remaining          : 779,425
Missing Values          : 0
Duplicate Records       : 0
Negative Quantities     : 0
Invalid Prices          : 0
Missing Customer IDs    : 0


In [19]:
# ==========================================
# Data Cleaning Audit - Exact Step Tracking
# ==========================================

audit = []

# Original dataset
audit.append({
    "Stage": "Original Dataset",
    "Rows": len(df)
})

# Remove duplicates
audit_df = df.drop_duplicates().copy()

audit.append({
    "Stage": "After Removing Duplicates",
    "Rows": len(audit_df)
})

# Remove missing Customer IDs
audit_df = audit_df.dropna(subset=["Customer ID"]).copy()

audit.append({
    "Stage": "After Removing Missing Customer IDs",
    "Rows": len(audit_df)
})

# Remove cancelled invoices
audit_df = audit_df[
    ~audit_df["Invoice"].astype(str).str.startswith("C")
].copy()

audit.append({
    "Stage": "After Removing Cancelled Invoices",
    "Rows": len(audit_df)
})

# Remove invalid quantities
audit_df = audit_df[
    audit_df["Quantity"] > 0
].copy()

audit.append({
    "Stage": "After Removing Invalid Quantities",
    "Rows": len(audit_df)
})

# Remove invalid prices
audit_df = audit_df[
    audit_df["Price"] > 0
].copy()

audit.append({
    "Stage": "After Removing Invalid Prices",
    "Rows": len(audit_df)
})

cleaning_audit = pd.DataFrame(audit)

cleaning_audit["Rows Removed"] = (
    cleaning_audit["Rows"].shift(1)
    - cleaning_audit["Rows"]
)

cleaning_audit

,Stage,Rows,Rows Removed
0,Original Dataset,1067371,NaN
1,After Removing Duplicates,1033036,34335.00
2,After Removing Missing Customer IDs,797885,235151.00
3,After Removing Cancelled Invoices,779495,18390.00
4,After Removing Invalid Quantities,779495,0.00
5,After Removing Invalid Prices,779425,70.00


In [21]:
# ==========================================
# Final Clean Dataset Profile
# ==========================================

final_profile = {
    "Rows": clean_df.shape[0],
    "Columns": clean_df.shape[1],
    "Unique Customers": clean_df["Customer ID"].nunique(),
    "Unique Products": clean_df["StockCode"].nunique(),
    "Unique Invoices": clean_df["Invoice"].nunique(),
    "Countries": clean_df["Country"].nunique(),
    "Total Revenue": clean_df["Revenue"].sum(),
    "Missing Values": clean_df.isnull().sum().sum(),
    "Duplicate Rows": clean_df.duplicated().sum()
}

final_profile_df = pd.DataFrame(
    final_profile.items(),
    columns=["Metric", "Value"]
)

final_profile_df

,Metric,Value
0,Rows,779425.00
1,Columns,10.00
2,Unique Customers,5878.00
3,Unique Products,4631.00
4,Unique Invoices,36969.00
5,Countries,41.00
6,Total Revenue,17374804.27
7,Missing Values,0.00
8,Duplicate Rows,0.00


In [20]:
# Save the Clean Dataset
clean_df.to_csv(r"C:\Users\Rizwan Hussain\Downloads\Compressed\cleaned_online_retail_II.csv", index=False)